# Анализ бизнес-сообществ ВКонтакте

Подготовка данных, проверка качества и первые рабочие сегменты для команды VK CaseChamp 2026. Исходные данные синтетические; результаты относятся к выборке, а не ко всем бизнесам VK. Здесь анализируется только лист «Бизнесы»; выводы показывают наблюдаемые связи и не доказывают причинный эффект.

## 1. Загрузка данных
Положите Excel рядом с ноутбуком либо измените `DATA_PATH`. Для исходного расположения автора предусмотрен запасной путь. Установите при необходимости: `pip install pandas openpyxl matplotlib`.

In [ ]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 80)
plt.rcParams['figure.figsize'] = (10, 4)

filename = 'VK_CaseChamp_2026_Отборочныи_этап_Датасет_1.xlsx'
candidates = [Path.cwd() / filename, Path.cwd().parent / filename]
DATA_PATH = next((p for p in candidates if p.exists()), candidates[0])
if not DATA_PATH.exists():
    raise FileNotFoundError(f'Не найден Excel: {DATA_PATH}. Положите файл рядом с ноутбуком или измените DATA_PATH.')
source_hash = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
df = pd.read_excel(DATA_PATH, sheet_name='Бизнесы')
print('Файл:', DATA_PATH)
print('SHA-256 исходного файла:', source_hash)
print('Строк:', len(df), '| столбцов:', len(df.columns))
display(df.head())

## 2. Структура, пропуски и проверка идентификаторов
Пропуск — не ноль. Проверки ниже показывают возможные проблемы и не удаляют наблюдения.

In [ ]:
quality = pd.DataFrame({'тип': df.dtypes.astype(str), 'пропусков': df.isna().sum(), 'пропусков, %': (df.isna().mean() * 100).round(2), 'уникальных значений': df.nunique(dropna=True)})
display(quality)
id_col = 'ID бизнеса'
print('Пустых ID:', df[id_col].isna().sum())
print('Повторных строк по ID:', df[id_col].duplicated().sum())
print('Полных дубликатов строк:', df.duplicated().sum())
print('Уникальных ID:', df[id_col].nunique())
display(df['Тип бизнеса (товары / услуги)'].value_counts(dropna=False).rename('бизнесов').to_frame())

In [ ]:
binary_cols = ['Подключён чекаут', 'Подключена доставка', 'Подключена оплата', 'Автор оригинального контента (оценка модели)', 'Участие в реферальной программе']
print('Значения бинарных полей:')
for col in binary_cols:
    print(col, df[col].value_counts(dropna=False).to_dict())
num = df.select_dtypes(include='number')
negative = (num < 0).sum()
print('Отрицательные значения по числовым полям:')
display(negative[negative > 0].rename('число строк').to_frame())
display(num.describe(percentiles=[.5, .9, .99]).T[['min', '50%', '90%', '99%', 'max']])

## 3. Сочетания для ручной проверки
Эти правила выделяют необычные сочетания, но сами по себе не доказывают ошибку: возможны особенности учёта. Товарные и обычные публикации могут считаться по-разному, поэтому не рассчитываем долю товарных публикаций до уточнения определений.

In [ ]:
checks = {
    'Активных товаров больше загруженных': df['Число активных товаров в каталоге'] > df['Товары загружены'],
    'Есть заказы, но чекаут не подключён': df['Заказы через чекаут'].gt(0) & df['Подключён чекаут'].eq(0),
    'Есть заказы, но GMV равен нулю': df['Заказы через чекаут'].gt(0) & df['GMV через чекаут, ₽'].eq(0),
    'GMV больше нуля, но заказов нет': df['GMV через чекаут, ₽'].gt(0) & df['Заказы через чекаут'].eq(0),
    'Товарных постов больше указанного числа постов': df['Посты с товаром за 30 дней'] > df['Посты за 30 дней'],
    'Товарных клипов больше указанного числа клипов': df['Клипы с товаром за 30 дней'] > df['Клипы за 30 дней'],
}
display(pd.DataFrame({'строк': {k: int(v.sum()) for k, v in checks.items()}, 'доля, %': {k: round(v.mean()*100, 2) for k, v in checks.items()}}))
for name, mask in checks.items():
    print(f'{name}: примеры ID:', df.loc[mask, id_col].head(5).tolist())

## 4. Признаки и журнал преобразований
Создаём признаки рядом с исходными полями; пропуски сохраняем. Необычные значения не исправляем. Заказы здесь — только заказы через чекаут, а не все возможные продажи бизнеса.

In [ ]:
data = df.copy()
def flag_positive(s):
    return s.gt(0).where(s.notna(), pd.NA).astype('boolean')

data['есть_активный_каталог'] = flag_positive(data['Число активных товаров в каталоге'])
data['есть_товарный_контент'] = (data['Посты с товаром за 30 дней'].gt(0) | data['Клипы с товаром за 30 дней'].gt(0)).where(data[['Посты с товаром за 30 дней', 'Клипы с товаром за 30 дней']].notna().all(axis=1), pd.NA).astype('boolean')
data['есть_заказы_через_чекаут'] = flag_positive(data['Заказы через чекаут'])
data['есть_расходы_на_рекламу'] = flag_positive(data['Траты на рекламу всего, ₽'])
for source, target in [('Подключён чекаут', 'есть_чекаут'), ('Подключена доставка', 'есть_доставка'), ('Подключена оплата', 'есть_оплата')]:
    data[target] = data[source].map({0: False, 1: True}).astype('boolean')
data['оригинальный_контент_состояние'] = data['Автор оригинального контента (оценка модели)'].map({0: 'нет', 1: 'да'}).fillna('нет оценки')
print('Состояние оценки оригинального контента:')
display(data['оригинальный_контент_состояние'].value_counts(dropna=False).to_frame('бизнесов'))
log = pd.DataFrame([
    ['Загрузка', 'Лист «Бизнесы», исходный файл не изменяется.'],
    ['Строки и исходные поля', 'Не удаляются, исходные значения не исправляются.'],
    ['Пропуски', 'Не заменяются нулями; для оценки оригинального контента выделено состояние «нет оценки».'],
    ['Каталог, заказы, реклама', 'Признак = значение больше нуля; при пропуске исходного значения признак неизвестен.'],
    ['Товарный контент', 'Хотя бы один товарный пост или клип; при пропуске любого из двух полей признак неизвестен.'],
    ['Инструменты', '0 = нет, 1 = да; прочие значения отображаются как неизвестные и требуют проверки.'],
    ['Сегменты', 'Последовательные правила ниже; услужные отдельно, неясные состояния не заполняются предположениями.'],
], columns=['шаг', 'правило'])
display(log)

## 5. Предварительные сегменты
Сегменты взаимоисключающие и описывают состояние, а не причину. Приоритет правил важен: заказы учитываются первыми, затем каталог и чекаут. Чисто услужные сообщества выделены отдельно, поскольку цель кейса — товары. Названия не утверждают, что бизнес никогда не продавал или не продавал вне VK.

In [ ]:
business_type = data['Тип бизнеса (товары / услуги)']
known_types = ['товарное', 'товарное и услужное', 'без каталога']
relevant_known = data[['есть_заказы_через_чекаут', 'есть_активный_каталог', 'есть_чекаут', 'есть_товарный_контент']].notna().all(axis=1)
data['сегмент'] = 'Неизвестные или неподдерживаемые значения'
eligible = business_type.isin(known_types) & relevant_known
orders = data['есть_заказы_через_чекаут'].fillna(False).astype(bool)
catalog = data['есть_активный_каталог'].fillna(False).astype(bool)
checkout = data['есть_чекаут'].fillna(False).astype(bool)
content = data['есть_товарный_контент'].fillna(False).astype(bool)
data.loc[business_type.eq('услужное'), 'сегмент'] = 'Услуги: отдельно'
data.loc[eligible & orders, 'сегмент'] = 'Есть заказы через чекаут'
data.loc[eligible & ~orders & catalog & checkout, 'сегмент'] = 'Каталог и чекаут, заказов нет'
data.loc[eligible & ~orders & catalog & ~checkout, 'сегмент'] = 'Каталог есть, чекаута нет'
data.loc[eligible & ~orders & ~catalog & content, 'сегмент'] = 'Нет каталога, есть товарный контент'
data.loc[eligible & ~orders & ~catalog & ~content, 'сегмент'] = 'Нет каталога и товарного контента'
assert len(data) == len(df) and data['сегмент'].notna().all()
assert data[id_col].equals(df[id_col])
print('Проверка покрытия сегментами:', len(data), 'из', len(df))
counts = data['сегмент'].value_counts(dropna=False).rename('бизнесов').to_frame()
counts['доля выборки, %'] = (counts['бизнесов'] / len(data) * 100).round(1)
display(counts)

In [ ]:
def group_summary(frame, group):
    result = frame.groupby(group, dropna=False).agg(
        бизнесов=(id_col, 'size'),
        с_заказами=('есть_заказы_через_чекаут', 'sum'),
        заказы=('Заказы через чекаут', 'sum'),
        gmv=('GMV через чекаут, ₽', 'sum'),
        подписчики_медиана=('Число подписчиков', 'median'),
        реклама_медиана=('Траты на рекламу всего, ₽', 'median'),
        товарные_просмотры_медиана=('Просмотры товарного контента', 'median'),
        каталог_доля=('есть_активный_каталог', 'mean'),
        чекаут_доля=('есть_чекаут', 'mean'),
        контент_доля=('есть_товарный_контент', 'mean'),
    )
    result['доля_с_заказами'] = result['с_заказами'] / result['бизнесов']
    result['заказов_на_продающий'] = result['заказы'] / result['с_заказами'].replace(0, np.nan)
    result['средний_чек_GMV_делить_на_заказы'] = result['gmv'] / result['заказы'].replace(0, np.nan)
    return result.sort_values('бизнесов', ascending=False)
segment_summary = group_summary(data, 'сегмент')
display(segment_summary.round(3))
print('Доли каталога, чекаута и контента рассчитаны по известным значениям; доля с заказами — по всем бизнесам группы. При пропусках заказов требуется уточнение знаменателя.')

## 6. Тематика, размер и первые сравнения
Это описательные срезы. Внутри сегментов есть смешение тематик и размеров; различия нельзя интерпретировать как эффект подключения инструмента. Сегменты построены в том числе по заказам, поэтому доля заказов в них не является независимым подтверждением качества сегментации.

In [ ]:
display(group_summary(data, 'Тематика').round(3))
display(group_summary(data, 'Тип бизнеса (товары / услуги)').round(3))
data['размер_аудитории'] = pd.cut(data['Число подписчиков'], bins=[-1, 999, 9999, 99999, float('inf')], labels=['до 1 тыс.', '1–10 тыс.', '10–100 тыс.', '100 тыс. и более'])
display(group_summary(data, 'размер_аудитории').round(3))
counts['бизнесов'].sort_values().plot.barh(title='Предварительные сегменты: число бизнесов', color='#3975b9')
plt.xlabel('Число бизнесов'); plt.tight_layout(); plt.show()

## 7. Контрольные показатели и выгрузка
Файлы сохраняются рядом с ноутбуком в папку `results`. Подготовленная таблица содержит все исходные столбцы и добавленные признаки; CSV записан в UTF-8 с BOM для Excel. Файл сегментов — предварительный и требует обсуждения с командой.

In [ ]:
controls = pd.DataFrame({'показатель': ['Строк', 'Уникальных ID', 'Сумма ID', 'Подписчики: сумма', 'Заказы через чекаут: сумма', 'GMV через чекаут: сумма, ₽', 'Пропусков оценки оригинального контента'],
    'значение': [len(data), data[id_col].nunique(), data[id_col].sum(), data['Число подписчиков'].sum(), data['Заказы через чекаут'].sum(), data['GMV через чекаут, ₽'].sum(), data['Автор оригинального контента (оценка модели)'].isna().sum()]})
display(controls)
assert data['Заказы через чекаут'].sum() == df['Заказы через чекаут'].sum()
assert data['GMV через чекаут, ₽'].sum() == df['GMV через чекаут, ₽'].sum()
assert data['Автор оригинального контента (оценка модели)'].isna().sum() == df['Автор оригинального контента (оценка модели)'].isna().sum()
out = Path('results'); out.mkdir(exist_ok=True)
data.to_csv(out / 'businesses_prepared.csv', index=False, encoding='utf-8-sig')
data[[id_col, 'сегмент']].to_csv(out / 'business_segments_draft.csv', index=False, encoding='utf-8-sig')
controls.to_csv(out / 'control_totals.csv', index=False, encoding='utf-8-sig')
log.to_csv(out / 'transformation_log.csv', index=False, encoding='utf-8-sig')
segment_summary.to_csv(out / 'segment_summary_draft.csv', encoding='utf-8-sig')
print('Созданы файлы:', *[p.name for p in sorted(out.glob('*.csv'))], sep='\n- ')
print('Проверьте результаты совместно с Юлианой; правила сегментов согласуйте с командой.')

## Что обсудить дальше

- Уточнить у организаторов определения общего и товарного числа публикаций, а также необычные сочетания значений.
- Вместе с Юлианой сверить число строк, ID, суммы и определения метрик.
- Проверить, можно ли для каждой рабочей группы предложить отдельное действие VK; при необходимости изменить правила.
- Сравнить похожие по тематике и числу подписчиков бизнесы, проверить чувствительность к крайним значениям. Модель машинного обучения для этого этапа не обязательна.

## 8. Каталог и чекаут: у скольких есть заказы

Рассматриваем бизнесы с активным каталогом и подключённым чекаутом. Доля считается от числа таких бизнесов с известным значением заказов. Нулевое значение означает отсутствие заказов **через чекаут за период**, а не отсутствие всех продаж. Категория «услужное» показана отдельно; чтобы сосредоточиться на товарах, ниже отдельно выводится выборка без неё.

In [ ]:
ready = data[data['есть_активный_каталог'].eq(True).fillna(False) & data['есть_чекаут'].eq(True).fillna(False)].copy()
ready['статус заказов'] = np.select([ready['Заказы через чекаут'].isna(), ready['Заказы через чекаут'].gt(0)], ['Неизвестно', 'Есть заказы'], default='Нет заказов')
for label, part in [('Все бизнесы', ready), ('Без чисто услужных бизнесов', ready[ready['Тип бизнеса (товары / услуги)'].ne('услужное')])]:
    known = part[part['статус заказов'].ne('Неизвестно')]
    result = part['статус заказов'].value_counts().reindex(['Есть заказы', 'Нет заказов', 'Неизвестно'], fill_value=0).to_frame('бизнесов')
    result['доля среди известных, %'] = (result['бизнесов'] / len(known) * 100).where(result.index != 'Неизвестно').round(1) if len(known) else np.nan
    print(label, '| всего:', len(part), '| с известным числом заказов:', len(known))
    display(result)
print('Тематики внутри группы с каталогом и чекаутом:')
display(pd.crosstab(ready['Тематика'], ready['статус заказов'], margins=True).sort_values('All', ascending=False))
print('Размеры аудитории внутри этой же группы:')
display(pd.crosstab(ready['размер_аудитории'], ready['статус заказов'], margins=True))

## 9. Тематики: объём и распространённость заказов

Сумма заказов зависит и от размера категории. Поэтому рядом считаем долю бизнесов с заказами и число заказов на продающий бизнес. Доли — от числа бизнесов с известным значением заказов; нулевой знаменатель оставляем пустым. Небольшие группы могут сильно меняться из-за отдельных наблюдений.

In [ ]:
def order_profile(frame, group):
    x = frame.copy()
    x['_известны'] = x['Заказы через чекаут'].notna().astype(int)
    x['_продают'] = x['Заказы через чекаут'].gt(0).astype(int)
    t = x.groupby(group, observed=True, dropna=False).agg(
        бизнесов=('ID бизнеса', 'size'), известны_заказы=('_известны', 'sum'),
        бизнесов_с_заказами=('_продают', 'sum'), заказов=('Заказы через чекаут', lambda s: s.sum(min_count=1)),
        gmv=('GMV через чекаут, ₽', lambda s: s.sum(min_count=1)))
    t['доля_с_заказами_%'] = 100 * t['бизнесов_с_заказами'] / t['известны_заказы'].replace(0, np.nan)
    t['заказов_на_продающий_бизнес'] = t['заказов'] / t['бизнесов_с_заказами'].replace(0, np.nan)
    return t.sort_values('заказов', ascending=False)

theme_profile = order_profile(data, 'Тематика')
display(theme_profile.round(2))
print('Для сравнения только товарных и смешанных бизнесов:')
tovar = data[data['Тип бизнеса (товары / услуги)'].isin(['товарное', 'товарное и услужное'])]
display(order_profile(tovar, 'Тематика').round(2))
print('Категория «без каталога» в этот товарный срез не входит: её товарный статус не определён.')

### Зависит ли результат от нескольких крупных бизнесов?

Для каждой тематики считаем долю заказов, приходящуюся на одного и на пять крупнейших продавцов, а также заказы после исключения крупнейшего продавца. Это проверка концентрации, а не исправление данных. При малом числе продающих бизнесов даже один продавец может сильно изменить картину.

In [ ]:
def concentration(g):
    orders = g['Заказы через чекаут'].dropna().sort_values(ascending=False)
    total = orders.sum()
    return pd.Series({'бизнесов': len(g), 'продающих': int(orders.gt(0).sum()), 'заказов': total,
                      'доля_топ1_%': 100 * orders.head(1).sum() / total if total > 0 else np.nan,
                      'доля_топ5_%': 100 * orders.head(5).sum() / total if total > 0 else np.nan,
                      'заказов_без_топ1': total - orders.head(1).sum() if len(orders) else np.nan})
concentration_table = data.groupby('Тематика', dropna=False)[['Заказы через чекаут']].apply(concentration).sort_values('заказов', ascending=False)
display(concentration_table.round(2))
print('Если доля топ-1 велика, общий результат тематики сильно зависит от одного бизнеса.')

## 10. Аудитория: сумма заказов и доля продающих

Сравниваем группы по числу подписчиков сообщества. Подписчики характеризуют аудиторию сообщества, но не позволяют определить размер бизнеса по выручке или числу сотрудников. Большая сумма заказов в группе может объясняться количеством бизнесов в ней.

In [ ]:
audience_profile = order_profile(data, 'размер_аудитории')
display(audience_profile.round(2))
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
audience_profile['заказов'].plot.bar(ax=axes[0], color='#3975b9', title='Заказы по размеру аудитории')
audience_profile['доля_с_заказами_%'].plot.bar(ax=axes[1], color='#58a47b', title='Бизнесы с заказами внутри группы, %')
for ax in axes: ax.set_xlabel('Подписчики'); ax.tick_params(axis='x', rotation=35)
plt.tight_layout(); plt.show()

## 11. Как превратить результаты в выводы для команды

Для каждого вывода запишите: **что видно в выборке → какое решение это помогает обсудить → чего мы не знаем**.

- **Каталог и чекаут без заказов:** можно исследовать поддержку подключившихся продавцов. Данные не показывают причину отсутствия заказов и продажи через другие каналы.
- **Высокие показатели конкретной тематики:** сравните размер группы, долю продающих и концентрацию. Название категории само по себе не доказывает, что бизнесы продают игровые ключи или являются мастерскими.
- **Высокий объём заказов в группе аудитории:** сопоставьте его с числом сообществ и долей продающих.

Гипотезу о пользе VK для мастерских и бизнеса с товарным контентом стоит проверять дополнительно: по одним названиям тематик нельзя определить состав бизнеса или причину покупок. Данные синтетические; связи не являются доказательством причинности.